# Time-Series Forecasting — Walk-Forward CV, Lag Features, and Baselines

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb16_time_series_forecasting_student.ipynb)


> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. Complete both to receive participation credit.

---


## Learning Objectives

By the end of this notebook, you will be able to:

1. Distinguish a forecasting problem from a generic supervised-learning problem and choose the right evaluation protocol.
2. Run a structured time-series EDA — time plot, seasonal sub-series, decomposition, autocorrelation — on a real labor-market dataset.
3. Build a **time-respecting** 60/20/20 train/validation/test split where the test window is the most recent slice of history.
4. Run **walk-forward cross-validation** with `TimeSeriesSplit` instead of k-fold CV (which would shuffle time and leak the future into the past).
5. Compare four classical forecasting benchmarks (Mean, Naive, Seasonal-Naive, Drift) against a learned **lag-feature linear regression** on identical CV folds.
6. Add **regularization** (Ridge) to the lag-feature linear model and decide whether it earns its place via the Student's *t* 95% CI overlap rule.
7. Open the locked test window in a **one-shot evaluation ceremony**, mirroring nb14's protocol but adapted to time.


## Setup

Import the libraries we will use across the notebook. Most of the toolkit is familiar from Week 1 — `pandas` for data wrangling, `matplotlib` and `seaborn` for plotting, `LinearRegression` and `Ridge` for modeling, `mean_absolute_error` for scoring, and `scipy.stats.t` for the Student's *t* CIs introduced in nb08. Two tools are new today: **`TimeSeriesSplit`** from `sklearn.model_selection`, which replaces `KFold` with a walk-forward splitter that never lets the future leak into the past, and **`STL`** from `statsmodels`, which decomposes a series into trend, seasonal, and remainder components. The `plot_acf` function (also from `statsmodels`) gives us the autocorrelation bar chart that will directly justify our lag-feature choices later.

In [ ]:
# Setup Cell
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from scipy.stats import t as student_t

warnings.filterwarnings("ignore")

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.precision', 3)
sns.set_style("whitegrid")

print("Setup complete!")
print(f"RANDOM_SEED = {RANDOM_SEED}")


**Reading the output:** A clean `Setup complete!` confirms all libraries loaded without errors. Both `statsmodels` and `sklearn` ship pre-installed with Google Colab, so you should not need to run `pip install` — if either fails, a simple `Runtime → Restart runtime` almost always resolves it. The `RANDOM_SEED = 474` and `figure.figsize = (10, 6)` settings match every prior notebook, so your outputs will reproduce exactly.

---

## 1. Why This Matters: Forecasting US Retail Employment

The **US Bureau of Labor Statistics** publishes monthly employment counts for every major industry. A **state-level workforce planner** uses those numbers to forecast next year's retail labor demand:

> *"I need a defensible forecast of US retail-sector employment one year out — with a confidence interval the legislature can read. Last year's headline number is not enough; I need the model and the diagnostics."*

This is **not** the kind of problem we solved in nb01–nb15. There, every row was an independent observation and a 60/20/20 random split was the right protocol. Here, the rows are months in a sequence — the **order matters**, and shuffling them would let the model peek at the future during training (a classic data leak). The fix is structural: the test window is always the **most recent** slice of history, and cross-validation walks forward in time.

This notebook ports the **Week-1 analytics workflow** (EDA → split → baselines → linear features → regularization) to the time-series setting, threading the same structural rule throughout: *the future cannot leak into the past.*

**A question that often comes up here:** *"Why isn't this just nb14 with a different metric?"* Two reasons. First, k-fold CV with shuffled rows would let row 50 be in the training fold and row 49 in the validation fold — the model would see a future month while learning to predict an earlier one. That defeats the entire idea of forecasting. Second, employment series have **seasonality** (holiday hiring) and **long-run trend** (decades of structural growth), so a feature engineered as "value 12 months ago" is structurally meaningful in a way that "row 12 in the dataset" is not.


## 2. Load the Data and Sanity Checks

We use the **US Employment dataset** from *Forecasting: Principles and Practice* (FPP3), the standard open-source textbook for time-series analysis. The dataset contains monthly employment counts (in thousands) for every major BLS industry classification from 1939 onward — roughly 80 years of monthly observations across dozens of industries. We filter to the **"Retail Trade"** series because it is the workforce planner's target and because it has all three structural features that make forecasting interesting: long-run trend, clear annual seasonality, and occasional recession-driven disruptions. The `ds` column is the date stamp; `y` is the employment count.

In [ ]:
# Load the full dataset (≈8.6 MB) directly from the course's GitHub raw URL
DATA_URL = (
    "https://raw.githubusercontent.com/davi-moreira/"
    "2026Summer_predictive_analytics_purdue_MGMT474/main/"
    "lecture_slides/08_time_series/data/us_employment.csv"
)
us_employment = pd.read_csv(DATA_URL, parse_dates=["ds"])

# Filter to the Retail Trade series and drop other columns
df = (
    us_employment.query('unique_id == "Retail Trade"')
    .loc[:, ["ds", "y"]]
    .sort_values("ds")
    .reset_index(drop=True)
)

print(f"Rows: {len(df):,}")
print(f"Date range: {df['ds'].min().date()}  ->  {df['ds'].max().date()}")
print(f"Missing values: {df.isna().sum().to_dict()}")
print()
print(df.head())


**Reading the output:**

You should see roughly **960 rows** spanning **1939-01 through 2019-09** (80 years of monthly data) with **zero missing values**. The two columns are `ds` (date stamp, parsed as `datetime64`) and `y` (employment in thousands). This is the cleanest possible time-series setup: a single numeric target indexed by a regular monthly date.

**A question that often comes up here:** *"Why is `unique_id` a string column?"* The original dataset is in long format — one row per (industry × month). The `unique_id` column tags each row with its industry. The lecture's `_08_time_series.ipynb` walks through several industries; we filter to a single one so the analysis stays focused.


## 3. Time-Series EDA — Six Plots, One Story

A time series asks for visual EDA before any modeling. The canonical sequence is: **time plot** (the whole series), **time plot zoomed** (a recent slice), **seasonal sub-series** (per-month box plot), **STL decomposition** (trend + seasonal + remainder), **ACF** (autocorrelation function), **lag-1 scatter** (do consecutive months track each other?). Six plots, one combined story.


### 3.1 Time plot — the whole series

The first plot every forecaster makes. Plot the full series on a timeline and let the shape speak: an upward or downward drift means **trend**, regular ripples mean **seasonality**, and sharp discontinuities mean **structural breaks** (recessions, policy changes). No statistical test conveys these features as quickly as a single well-drawn time plot — and the workforce planner who skips this step risks building a model that ignores structure the eye catches in seconds.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["ds"], df["y"], color="#1f77b4", linewidth=0.8)
ax.set_xlabel("Year")
ax.set_ylabel("Employment (thousands)")
ax.set_title("US Retail Trade Employment, 1939–2019 (monthly)")
plt.tight_layout()
plt.show()


**Reading the output:**

Three structural components are visible in this single plot, and the workforce planner needs to name all three before modeling begins.

**1. Long-run trend.** Employment roughly **triples** from about 5,000 thousand in 1939 to over 15,000 thousand by 2019. The growth is not uniform — you can trace the mid-century post-war expansion, the 1990s retail and e-commerce boom, and the gradual recovery after 2010. The overall shape is an upward curve that any forecasting model must track; a model that ignores it would predict the 1950s level for 2020.

**2. Seasonality.** Look closely and you will see small annual ripples running along the trend line — the curve is not smooth but gently serrated. Those ripples are **holiday retail hiring** in November and December (the peaks) and **post-holiday layoffs** in January and February (the troughs). At this 80-year zoom level the ripples are hard to read, which is exactly why the next plot zooms in. But even here, the regularity is visible: the same up-down pattern repeats every 12 months for eight decades. A model that captures trend but ignores seasonality will systematically over-predict in January and under-predict in December.

**3. Structural breaks.** The sharpest disruptions are **recessions**: the 2008–2010 financial crisis is the most dramatic (a steep drop of roughly 2,000 thousand employees followed by a multi-year recovery), but smaller dips are visible in 1974 (oil crisis), 1980 (double-dip recession), 1990, and 2001 (dot-com bust). These breaks are driven by macroeconomic shocks that the series itself cannot predict — no lag feature or seasonal pattern will warn you that a financial crisis is coming. The practical implication for the workforce planner: the model's prediction interval must be wide enough to cover these tail events, and any forecast delivered to the legislature should carry a caveat about recession-driven uncertainty.

A single forecasting model has to capture trend and seasonality (the learnable components) while honestly acknowledging that structural breaks will occasionally push actuals outside even a well-calibrated prediction interval.

### 3.2 Time plot zoomed — last 10 years

The full 80-year time plot compresses 960 monthly observations into a single curve, which makes the annual seasonal cycle nearly invisible — the ripples are too small relative to the 80-year trend. Zooming in to the last decade stretches the x-axis enough to see individual holiday peaks and post-holiday dips. This is the plot that tells the workforce planner *how much* staffing swings within a single year.

In [ ]:
recent = df[df["ds"] >= "2010-01-01"]
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(recent["ds"], recent["y"], "o-", color="#2ca02c", markersize=4)
ax.set_title("US Retail Trade Employment — 2010–2019 (monthly)")
ax.set_xlabel("Year")
ax.set_ylabel("Employment (thousands)")
plt.tight_layout()
plt.show()


**Reading the output:**

The zoomed plot makes the seasonal cycle unmistakable. Three features are now readable that the 80-year plot compressed into illegibility.

**Annual peaks in November/December.** Every year, employment surges as retailers staff up for the holiday season — Black Friday through Christmas. The peaks are the tallest points in each annual wave. For the workforce planner, these peaks are not surprises; they are predictable staffing events that drive temporary-hire budgets, training timelines, and warehouse capacity decisions months in advance.

**Post-holiday troughs in January/February.** After the holidays, seasonal positions end and employment drops sharply. The trough is typically the lowest point of the annual cycle. The gap between the December peak and the January trough — a swing of several percent of the total retail workforce — is the seasonal amplitude the model needs to capture.

**Slow upward trend underneath the waves.** Each year's trough is slightly higher than the previous year's trough; each peak is slightly higher than the previous peak. That is the long-run trend visible at this scale — the same structural growth that §3.1's full time plot showed over 80 years, now legible as a gentle upward tilt beneath the seasonal ripples.

This seasonal cycle is exactly the structure that a **lag-12 feature** (this month vs. the same month last year) will capture automatically in section 8. The model does not need to "know" about holiday hiring — it just needs to see that December 2018 looked a lot like December 2017.

### 3.3 Seasonal sub-series box plot

A box plot of `y` grouped by **month-of-year** compresses 80 Decembers into one box, 80 Januaries into another, and so on. It answers two questions at once: *"how strong is the seasonal pattern?"* (do the medians differ across months?) and *"how stable is it?"* (are the boxes tight or sprawling?). If December's box sits consistently above June's across 80 years, the seasonal effect is real and worth modeling. If every month's box overlaps every other, there is no seasonality to capture.

In [ ]:
df_seasonal = df.copy()
df_seasonal["month"] = df_seasonal["ds"].dt.month
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df_seasonal, x="month", y="y", ax=ax, color="#ff7f0e")
ax.set_title("Seasonal sub-series — employment distribution by month-of-year")
ax.set_xlabel("Month of year")
ax.set_ylabel("Employment (thousands)")
plt.tight_layout()
plt.show()


**Reading the output:**

The box plot answers both seasonal questions at once.

**How strong is the seasonal pattern?** Strong enough to act on. **December** sits visibly above every other month — this is the holiday retail hiring surge at its clearest. November comes next (early holiday ramp-up and pre-Black Friday staffing), then a steady plateau from March through October, then the **January/February dip** as seasonal positions end and post-holiday returns wind down. The medians trace a smooth annual cycle that the workforce planner can use as a staffing calendar: expect the highest labor demand in December, the lowest in January, and a stable mid-range from spring through early fall.

**How stable is the seasonal pattern?** Stable in shape, but the boxes are wide. That width is not noise — it is the **long-run trend hiding inside the by-month aggregation**. Think about what a single "December" box contains: December 1945 (roughly 6,000 thousand employees) and December 2015 (roughly 16,000 thousand). Both are Decembers, but they sit at very different absolute levels because the economy grew over those 70 years. When you lump them into one box, the box stretches from 6,000 to 16,000 — a range driven by the trend, not by December-to-December instability. The seasonal *shape* (December > November > ... > January) is real and consistent across decades; the seasonal *amplitude* relative to the trend is moderate.

This distinction — real seasonal shape but trend-inflated box width — is why the STL decomposition in the next plot is valuable. STL separates the trend from the seasonal component mathematically, so you can see each one's contribution cleanly.

### 3.4 STL decomposition — trend + seasonal + remainder

**STL (Seasonal-Trend decomposition using Loess)** splits the series into three additive components:

$$y_t = T_t + S_t + R_t$$

— a smooth trend, a periodic seasonal pattern, and what is left over (the "remainder"). It is the time-series analog of "explain the variance and look at the residuals."


In [ ]:
# STL decomposition with monthly period
ts = df.set_index("ds")["y"]
stl = STL(ts, period=12, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
axes[0].plot(ts.index, ts.values, color="black"); axes[0].set_ylabel("y (data)")
axes[1].plot(ts.index, stl.trend, color="#1f77b4"); axes[1].set_ylabel("Trend")
axes[2].plot(ts.index, stl.seasonal, color="#2ca02c"); axes[2].set_ylabel("Seasonal")
axes[3].plot(ts.index, stl.resid, color="#d62728"); axes[3].set_ylabel("Remainder")
axes[3].axhline(0, color="black", linewidth=0.5)
axes[0].set_title("STL Decomposition — US Retail Trade Employment")
plt.tight_layout()
plt.show()


**Reading the output:**

Four panels, top to bottom, each isolating one component of the series.

**Data (top panel).** The raw series — the same curve you saw in §3.1's time plot. It contains all three structural components mixed together.

**Trend (second panel).** The smooth long-run component after the seasonal and residual fluctuations have been stripped away. You can now see the structural growth trajectory clearly: steady expansion from the 1940s through the 1970s, a plateau and mild dips around the oil-crisis recessions, accelerating growth through the 1990s retail boom, the sharp **2008 financial-crisis drop** (roughly 2,000 thousand employees lost in two years), and the gradual post-2010 recovery. This is the component that lag-1 features will capture — each month's employment level is closely related to last month's.

**Seasonal (third panel).** The regular annual pattern, isolated from the trend. The same wave shape repeats every 12 months for 80 years — December peaks, January troughs, a consistent amplitude throughout. Notice that the wave height does not grow over time even though the trend does; this confirms the **additive** decomposition was the right choice. If the seasonal swings had grown proportionally with the level (bigger absolute swings at higher employment), a multiplicative decomposition would have been needed instead.

**Remainder (bottom panel).** What neither trend nor seasonality explains. Most of the time, the remainder fluctuates in a narrow band around zero — the trend and seasonal components account for nearly all of the variation. But there are visible **spikes around recessions**: the 2008 crisis produces the largest residual, and smaller spikes appear in 1974, 1980, 1990, and 2001. These are the structural breaks from §3.1 — macroeconomic shocks that arrive from outside the series. No lag feature or seasonal pattern will predict them; the prediction interval in §10 will need to be wide enough to accommodate them.

> **A question that often comes up here:** *"Should I use additive or multiplicative decomposition?"* Additive when the seasonal amplitude does not grow with the trend; multiplicative when it does. Visually, the seasonal swing here is roughly the same height in 1950 (small absolute employment) as in 2010 (large absolute employment) → additive is the right call. If the seasonal swing was larger in absolute terms when employment was higher, multiplicative would fit better.

### 3.5 Autocorrelation (ACF) plot

The ACF asks *"how strongly does month $t$'s value depend on month $t-k$'s value, for each lag $k$?"*. Each bar in the plot is a correlation coefficient between the series and a lagged copy of itself. Tall bars at lags 1, 2, 3 mean **trend and momentum** — recent months are highly correlated with the present. A tall bar at lag 12 (and again at 24) means **annual seasonality** — what happened a year ago is a strong predictor of what happens now. The ACF is the empirical tool that tells us *which lags are worth engineering as features*.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
plot_acf(ts.values, lags=36, ax=ax, zero=False)
ax.set_title("Autocorrelation function — lags 1 through 36")
ax.set_xlabel("Lag (months)")
plt.tight_layout()
plt.show()


**Reading the output:**

The ACF translates the visual patterns from the first four plots into precise numerical evidence about which lags carry predictive signal.

**Slow decay across lags 1–24.** The bars start tall at lag 1 (correlation above 0.95) and decline gradually through lag 24. This slow decay is the **signature of a strong trend** — when employment is high this month, it was also high last month, and the month before that, and so on. The practical implication: the most recent value (`lag1`) is by far the single most informative predictor of the next value. This is why the naive forecast ("next month = last month") will be so hard to beat.

**Local peaks at lags 12 and 24.** On top of the slow decay, the bars at lags 12 and 24 are visibly taller than their neighbors. Lag 12 means "this month correlates strongly with the same month one year ago" — that is the annual seasonal cycle the zoomed time plot and the box plot already showed. Lag 24 means the same pattern holds two years back. These peaks are weaker than lag 1 (the trend dominates), but they carry **independent seasonal information** that lag 1 alone cannot provide.

**The blue shaded band** marks the 95% confidence interval for "no significant autocorrelation." Every bar that extends beyond the band is statistically significant. On this series, every bar through lag 36 is significant — the series has strong, persistent structure at every timescale up to three years.

This ACF gives us **direct empirical justification** for the two lag features we will engineer in section 8: `lag1` captures the trend and short-term momentum (the slow decay), and `lag12` captures the annual seasonal cycle (the lag-12 peak). The ACF is the diagnostic; the features are the response.

### 3.6 Lag-1 scatter

The simplest forecast in the world is *"next month equals last month."* The lag-1 scatter plots this month's employment against last month's — every dot is one month. If the dots hug the 45° line, the naive forecast is strong (month-to-month changes are small). If they scatter into a cloud, consecutive months are volatile and the naive forecast will have large errors. For the workforce planner, this plot answers a practical question: *"can I get away with just using last month's number, or do I genuinely need a model?"*

In [ ]:
lag1_df = df.assign(lag1=df["y"].shift(1)).dropna()
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(lag1_df["lag1"], lag1_df["y"], s=8, alpha=0.5, color="#9467bd")
lo, hi = lag1_df["y"].min(), lag1_df["y"].max()
ax.plot([lo, hi], [lo, hi], "k--", linewidth=0.8, label="Perfect lag-1 forecast (y = lag1)")
ax.set_xlabel("Employment, t-1 (lag1)")
ax.set_ylabel("Employment, t")
ax.set_title("Lag-1 scatter — does the previous month predict the current month?")
ax.legend()
plt.tight_layout()
plt.show()


**Reading the output:**

Every dot in this scatter is one month. The x-coordinate is last month's employment; the y-coordinate is this month's. The dashed diagonal is the 45° line — the line where "this month = last month" exactly.

**The dots hug the 45° line tightly.** Month-to-month changes in retail employment are small relative to the overall level. A month at 15,000 thousand employees is almost always followed by a month between 14,800 and 15,200 — a change of at most 1–2%. That tightness is the visual proof that the **naive forecast** ("next month = last month") will be a strong baseline. Any learned model that does not beat it is not earning its keep.

**The cloud is elongated, not round.** The scatter stretches from the lower-left (early decades, lower employment) to the upper-right (recent decades, higher employment). That elongation is the trend — the series moves through different employment regimes over 80 years. Within each regime, the dots cluster tightly around the diagonal.

**There are no dramatic outliers far from the line.** Even the recession months (2008–2009) do not produce dots that land far off the diagonal, because the employment drops happened over multiple months rather than in a single catastrophic jump. This is good news for the lag-feature regression: the relationship between consecutive months is approximately linear and stable.

> **A question that often comes up here:** *"Does this mean the naive forecast will always be hard to beat?"* For slow-moving, strongly autocorrelated series like monthly employment, yes — the naive forecast inherits most of the signal for free. But for volatile series — daily tech stocks, hourly web traffic, cryptocurrency — the lag-1 scatter would show a much wider cloud, and the naive baseline would have much larger errors. The tighter the scatter hugs the 45° line, the higher the bar any learned model has to clear.

Six plots, one combined story: strong trend, clear annual seasonality, and tight lag-1 autocorrelation. Section 4 translates those structural facts into the single rule that governs every modeling decision below.

---

## 4. The Structural Rule — Never Shuffle, Never Leak the Future

Every static-classification rule we built since nb01 still works in this notebook — **except one**. Rows here are months in a sequence, and shuffling them would let the model peek at the future during training. That single structural change cascades into three downstream changes:

1. **Train/test split**: the test window is the **most recent slice** of history, not a random sample.
2. **Cross-validation**: every fold's training data must come strictly **before** its validation data.
3. **Features**: lag features (last month, 12 months ago) replace random feature engineering.

Sections 5–7 implement each one in order.

---


## 5. Time-Respecting 60/20/20 Split

Week 1 taught the 60/20/20 split as a random partition. Today we use the same proportions, but the partition is **order-respecting**: the oldest 60% of the history becomes the training set, the middle 20% becomes validation, and the most recent 20% becomes the **locked test window**. No shuffling — every row stays in its chronological position. The workforce planner's credibility depends on this discipline: a forecast model that was fitted on data from 2010 and tested on data from 1990 proves nothing about its ability to predict the future. The test window is touched exactly once, in section 10, after the champion is chosen.

In [ ]:
n = len(df)
n_train = int(n * 0.60)
n_val = int(n * 0.20)
n_test = n - n_train - n_val

df_train = df.iloc[:n_train].copy()
df_val   = df.iloc[n_train:n_train + n_val].copy()
df_test  = df.iloc[n_train + n_val:].copy()

print(f"Train: {df_train['ds'].min().date()} -> {df_train['ds'].max().date()}  (n={len(df_train)})")
print(f"Val  : {df_val['ds'].min().date()} -> {df_val['ds'].max().date()}  (n={len(df_val)})")
print(f"Test : {df_test['ds'].min().date()} -> {df_test['ds'].max().date()}  (n={len(df_test)})  [LOCKED]")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_train["ds"], df_train["y"], color="#1f77b4", label="Train")
ax.plot(df_val["ds"],   df_val["y"],   color="#ff7f0e", label="Val")
ax.plot(df_test["ds"],  df_test["y"],  color="#d62728", linestyle="--", label="Test (locked)")
for boundary, color in [(df_train["ds"].max(), "grey"), (df_val["ds"].max(), "grey")]:
    ax.axvline(boundary, color=color, linestyle=":", alpha=0.7)
ax.set_title("Time-Respecting 60/20/20 Split")
ax.legend()
plt.tight_layout()
plt.show()


**Reading the output:**

The plot shows the full 80-year series divided into three chronological segments, separated by vertical grey dashed lines.

**Blue (Train, \~60%)** covers the earliest portion of the history — roughly 1939 through the late 1980s, about 576 months. This is the data the model learns from: all the fitting, all the cross-validation, all the feature engineering happens here. The trend, seasonality, and structural breaks from §3's EDA are all present in this window.

**Orange (Validation, \~20%)** covers the next portion — roughly the late 1980s through the early 2000s, about 192 months. This is the window used for single-split evaluation and for building the walk-forward CV folds in §6. It is recent enough to reflect modern retail dynamics but old enough that the locked test window still captures the most recent period.

**Dashed red (Test, \~20%, LOCKED)** covers the most recent portion — roughly the early 2000s through 2019, about 192 months. This window includes the 2008 recession and the post-crisis recovery — the hardest structural break in the dataset. It stays sealed until the section-10 ceremony, exactly as the test set stayed sealed through nb08–nb14 for static classification.

Every model-selection decision below uses only the blue and orange portions. The dashed red is the one-shot evaluation that tells the workforce planner whether the champion generalizes to truly unseen time.

> **A question that often comes up here:** *"Why 60/20/20 instead of, say, 80/10/10?"* Same reason as nb01: 60% gives the model enough history to fit, 20% gives validation enough power to discriminate between candidates with non-overlapping CIs, and 20% locked test gives the final ceremony enough rows to be a meaningful sample of the recent dynamics. The exact split is conventional; the discipline of holding out a recent slice is structural.

The split defines *which* rows go where; section 6 defines *how* to evaluate models honestly on the training portion using walk-forward cross-validation.

---

## 6. Walk-Forward Cross-Validation with `TimeSeriesSplit`

In nb08, `KFold` shuffled rows into training and validation folds — fine when rows are independent. Here, shuffling would let the model see future months while training on earlier ones, defeating the purpose of forecasting. **`TimeSeriesSplit(n_splits=5)`** is the structural fix: it produces 5 folds where every fold's training data comes strictly **before** its validation data, and the training window grows with each fold (an expanding-window design). This mimics real-world deployment, where you retrain monthly on an ever-growing history and always forecast forward into unseen time.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

fig, ax = plt.subplots(figsize=(11, 4))
for fold, (train_idx, val_idx) in enumerate(tscv.split(df_train)):
    ax.plot(train_idx, [fold]*len(train_idx), "s", color="#1f77b4", markersize=3, label="train" if fold == 0 else "")
    ax.plot(val_idx,   [fold]*len(val_idx),   "s", color="#ff7f0e", markersize=3, label="val" if fold == 0 else "")
ax.set_yticks(range(5))
ax.set_yticklabels([f"fold {i+1}" for i in range(5)])
ax.set_xlabel("Month index in training data")
ax.set_title("Walk-Forward CV: train (blue) always precedes val (orange)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


**Reading the output:**

The visualization shows five rows — one per CV fold — with blue squares marking training months and orange squares marking validation months.

**Fold 1** (top row) has the smallest training window and the first validation window. The model sees only the earliest portion of the training data and is tested on the months that immediately follow. This is the hardest fold — the model has the least history to learn from.

**Fold 5** (bottom row) has the largest training window — roughly five times as much data as fold 1 — and the last validation window. This is the easiest fold and the one closest to what deployment looks like: the model has nearly all the available training history.

Two structural facts to carry forward. First, **orange always sits to the right of blue** — the model never sees a future month while learning to predict an earlier one. That is the time-respecting constraint that makes walk-forward CV honest. Second, **the training window grows** across folds. That growth is not a bug; it mimics real-world deployment, where you retrain monthly on an ever-growing history and always forecast into unseen time. The consequence is that earlier folds are harder and later folds are easier — which is why the per-fold MAEs in §9 will show some variation.

> **A question that often comes up here:** *"Does the growing window give later folds an unfair advantage?"* Yes, and that is realistic. In deployment, you always have more history than you did six months ago. The expanding-window design captures that asymmetry honestly. A fixed-size sliding window (where you drop the oldest rows as you add new ones) is an alternative when you believe only recent history is relevant — but for a series with an 80-year trend, throwing away the early decades would discard useful signal.

With the walk-forward folds in hand, the next question is what to measure on each fold. Section 7 introduces four forecasting metrics and runs the classical benchmarks under all of them.

---

## 7. Forecasting Metrics — Four Benchmarks Under Four Lenses

Before evaluating any model, decide what *good* means. Four metrics show up in every time-series textbook; each answers a slightly different question about forecast quality. Sub-section 7.1 defines each metric with its formula; 7.2 gives a quick "when to use which" guide; the code cell that follows runs the four classical benchmarks under all four metrics on the validation window.


### 7.1 The four metrics

**Mean Absolute Error (MAE).**

$$\text{MAE} = \frac{1}{n}\sum_{t=1}^{n} \left| y_t - \hat{y}_t \right|$$

In the data's original units. Robust to outliers but treats large and small errors equally.

**Root Mean Squared Error (RMSE).**

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{t=1}^{n} \left( y_t - \hat{y}_t \right)^2}$$

Squaring before averaging penalizes large errors more than small ones. Sensitive to outliers; not in the data's original units (close-ish — same order of magnitude).

**Mean Absolute Percentage Error (MAPE).**

$$\text{MAPE} = \frac{100}{n}\sum_{t=1}^{n} \left| \frac{y_t - \hat{y}_t}{y_t} \right|$$

A scale-free percentage, comparable across series. Breaks if any $y_t \approx 0$ and is asymmetric — over-forecasts hurt more than under-forecasts of equal magnitude.

**Mean Absolute Scaled Error (MASE).**

$$\text{MASE} = \frac{\text{MAE}}{\text{MAE}_{\text{seasonal-naive, in-sample}}}$$

Scale-free; benchmarks against the in-sample seasonal-naive baseline. A value below 1 means the model beats the free baseline; above 1 means the seasonal-naive forecast is better than your model.


### 7.2 When to use which

| Use this metric ... | ... when |
|---|---|
| **MAE** | Reporting in business units (employees, dollars, units sold) and you want a robust default. |
| **RMSE** | Large misses cost much more than small ones (stock-outs, surge planning, safety-critical capacity). |
| **MAPE** | Reporting to non-technical audiences ("we are off by 3% on average") — and only when *y* is far from zero across the validation window. |
| **MASE** | Comparing forecasts across multiple series with different scales (cross-region demand, multi-product KPIs). |

**A question that often comes up here:** *"If the metrics rank models differently, which do I trust?"* The one whose error structure matches your business cost. If a stock-out costs 10× as much as overstock, RMSE is the honest metric — squaring penalizes the rare large miss exactly the way the cost matrix does. There is no "best" metric in the abstract; there is only the metric that aligns with consequences.


The four classical benchmarks are simple enough to implement by hand — that simplicity is the point. **Mean** forecasts the historical average for every future period (a flat line that ignores trend and seasonality entirely). **Naive** carries the last observed value forward unchanged, betting that tomorrow looks like today. **Seasonal-Naive** replays the most recent complete season (here, the last 12 months) forward, betting that next January looks like last January. **Drift** draws a straight line from the first training observation to the last and extends it forward — a simple trend extrapolation. The code cell below implements all four and evaluates each under the four metrics defined above.

In [ ]:
def forecast_mean(history, h):
    return np.full(h, history.mean())

def forecast_naive(history, h):
    return np.full(h, history.iloc[-1])

def forecast_seasonal_naive(history, h, season=12):
    last_season = history.iloc[-season:].values
    return np.tile(last_season, int(np.ceil(h / season)))[:h]

def forecast_drift(history, h):
    slope = (history.iloc[-1] - history.iloc[0]) / (len(history) - 1)
    return history.iloc[-1] + slope * np.arange(1, h + 1)

# Single-shot forecast on the validation window
horizon = len(df_val)
hist = df_train["y"]
preds = {
    "Mean":            forecast_mean(hist, horizon),
    "Naive":           forecast_naive(hist, horizon),
    "Seasonal-Naive":  forecast_seasonal_naive(hist, horizon),
    "Drift":           forecast_drift(hist, horizon),
}

# Plot all four against the validation actuals
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_train["ds"].iloc[-60:], df_train["y"].iloc[-60:], color="black", label="Train (last 60 mo)")
ax.plot(df_val["ds"], df_val["y"], color="black", linestyle="--", label="Val (actual)")
for name, p in preds.items():
    ax.plot(df_val["ds"], p, label=name)
ax.set_title("Four classical benchmarks on the validation window")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

# --- Multi-metric evaluation utility ---
def all_metrics(y_true, y_pred, training_y, season=12):
    """Return MAE, RMSE, MAPE, MASE for one (y_true, y_pred) pair."""
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    mae = np.mean(np.abs(yt - yp))
    rmse = np.sqrt(np.mean((yt - yp) ** 2))
    mape = np.mean(np.abs((yt - yp) / yt)) * 100.0
    th = np.asarray(training_y, dtype=float)
    seasonal_naive_errors = np.abs(th[season:] - th[:-season])
    mae_naive = seasonal_naive_errors.mean()
    mase = mae / mae_naive if mae_naive > 0 else np.nan
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "MASE": mase}

# Evaluate each benchmark under all four metrics on the validation window
benchmark_table = pd.DataFrame({
    name: all_metrics(df_val["y"].values, p, df_train["y"].values)
    for name, p in preds.items()
}).T
print("Four classical benchmarks — single-shot validation, all four metrics:")
print(benchmark_table.round(3))

print("\\nRanking under each metric (1 = best):")
print(benchmark_table.rank(axis=0).astype(int))


**Reading the output:**

Three outputs to read in sequence: the plot, the metric table, and the ranking table.

**The plot** shows five lines crossing the validation window: the four baseline forecasts and the black dashed actual values. Look for which colored line tracks the black actuals most closely. On US Retail Trade, **Seasonal-Naive** (which replays the last 12 months of training forward) typically hugs the actuals best — it captures the annual peaks and troughs that the other baselines miss. **Mean** is visibly the worst — it draws a flat horizontal line through the middle of the validation window, missing both the trend and the seasonality. **Naive** (carry the last training value forward) captures the level but misses the seasonal swing. **Drift** (straight-line extrapolation) captures the trend direction but also misses the seasonality.

**The metric table** puts numbers behind the visual impression. Four rows (one per baseline), four columns (MAE, RMSE, MAPE, MASE). Read the MAE column first — it is in the planner's native units (thousands of employees). Seasonal-Naive typically has the lowest MAE, confirming what the plot showed. The MASE column is the reality check: a MASE below 1.0 means the baseline beats the in-sample seasonal-naive benchmark; above 1.0 means it lost. By definition, Seasonal-Naive's MASE is near 1.0 (it *is* the reference baseline for that metric).

**The ranking table** shows where the four metrics agree and where they disagree. When all four metrics rank the same baseline first, the ranking is **robust** — you can state the winner with confidence. When rankings disagree (say, MAE picks Seasonal-Naive but RMSE picks Drift), the disagreement is the teaching moment: MAE treats every error equally, while RMSE punishes the occasional large miss more heavily. The choice between them is a **business** choice — does the workforce planner's procurement plan tolerate steady small misses (favor MAE) or is a single large miss catastrophic (favor RMSE)?

These rankings depend on the data's structural features. The next subsection makes that concrete by running the same benchmarks on a series with no seasonality at all.

---

### 7.3 Non-Seasonal Contrast — Google Daily Stock Prices

The US Retail Trade series above has clear annual seasonality, which is why **Seasonal-Naive** was such a strong baseline. But many business series — daily stock prices, intraday traffic, hourly server load, web-conversion rates — have little or no calendar seasonality. The classical benchmarks behave very differently there: **Drift** (linear extrapolation from start to end of training) often beats Seasonal-Naive by a lot because there is no annual pattern to lean on, and **Naive** (carry the last training value forward) can also be competitive because consecutive days are highly correlated.

To make the contrast concrete, we run the same benchmarks on Google daily closing prices: train on 2015, test on January 2016. The pedagogical point is that **the choice of benchmark depends on the data’s structural features, not on the model**.

In [ ]:
# Load Google daily closing prices from the GAFA stock dataset (same
# long-format columns as the employment data: unique_id, ds, y).
GAFA_URL = (
    "https://raw.githubusercontent.com/davi-moreira/"
    "2026Summer_predictive_analytics_purdue_MGMT474/main/"
    "lecture_slides/08_time_series/data/gafa_stock.csv"
)
gafa = pd.read_csv(GAFA_URL, parse_dates=["ds"])
goog = (
    gafa[gafa["unique_id"] == "GOOG_Close"]
    .loc[:, ["ds", "y"]]
    .sort_values("ds")
    .reset_index(drop=True)
)

# Train: 2015 (full calendar year). Test: January 2016.
goog_train = goog[(goog["ds"] >= "2015-01-01") & (goog["ds"] <  "2016-01-01")]
goog_test  = goog[(goog["ds"] >= "2016-01-01") & (goog["ds"] <  "2016-02-01")]
print(f"Train: {goog_train['ds'].min().date()} -> {goog_train['ds'].max().date()}  (n={len(goog_train)})")
print(f"Test : {goog_test['ds'].min().date()} -> {goog_test['ds'].max().date()}  (n={len(goog_test)})")

# Three classical benchmarks. Seasonal-Naive omitted because daily stock prices
# have no calendar seasonality (no 12-period annual cycle to anchor against).
horizon_g = len(goog_test)
hist_g = goog_train["y"]
preds_g = {
    "Mean":  forecast_mean(hist_g, horizon_g),
    "Naive": forecast_naive(hist_g, horizon_g),
    "Drift": forecast_drift(hist_g, horizon_g),
}

# Plot: 2015 training in grey + Jan 2016 test in black + three forecasts as
# colored lines extending past the training cutoff.
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(goog_train["ds"], goog_train["y"],
        color="grey", alpha=0.6, linewidth=0.8, label="2015 (train)")
ax.plot(goog_test["ds"],  goog_test["y"],
        color="black", linewidth=1.5, label="Jan 2016 (test, actual)")
for name, p in preds_g.items():
    ax.plot(goog_test["ds"], p, label=name, linewidth=1.2)
ax.axvline(pd.Timestamp("2016-01-01"), color="red", linestyle=":", alpha=0.5,
           label="Train / test boundary")
ax.set_title("Google Daily Closing Price — 2015 Train, Jan 2016 Test")
ax.set_xlabel("Date")
ax.set_ylabel("Closing price (USD)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

# Accuracy table — use season=1 (next-step random-walk baseline) for MASE
# because the series has no annual seasonality.
goog_table = pd.DataFrame({
    name: all_metrics(goog_test["y"].values, p, goog_train["y"].values, season=1)
    for name, p in preds_g.items()
}).T
print("\nAccuracy table — Google daily, January 2016 horizon (MASE referenced to 1-step naive):")
print(goog_table.round(3))
print("\nRanking under each metric (1 = best):")
print(goog_table.rank(axis=0).astype(int))

**Reading the output:**

The ranking on Google daily prices typically **flips** relative to US Retail Trade — a concrete demonstration that the right baseline depends on the data, not on a modeling assumption.

**In the plot**, the 2015 training data appears in grey and the January 2016 test window in black. The three forecast lines extend past the red vertical cutoff. **Drift** (the straight-line extrapolation from the first to the last training price) typically tracks the January actuals most closely — it captures the mild upward or downward momentum of the 2015 price trajectory. **Naive** (carry the last December 2015 closing price forward) draws a flat line that is competitive because consecutive trading days are highly correlated; tomorrow's price is almost always close to today's. **Mean** (the average of all 2015 closing prices) is usually the worst — it projects a price from mid-2015 into January 2016, throwing away the level the stock actually closed at.

**Why the flip?** US Retail Trade has strong annual seasonality, so Seasonal-Naive — which replays last year's pattern — is the natural winner. Google daily stock has no calendar seasonality (there is no "December peak" in closing prices), so Seasonal-Naive is not even in the race. The dominant structure is a slow random-walk-like drift, which is exactly what the Drift baseline captures.

**MASE with season = 1** deserves a note. For the retail employment series, MASE used the 12-step seasonal-naive baseline as the denominator. For a non-seasonal series, the natural denominator is the 1-step naive baseline (a random walk). The formula is the same; only the reference baseline changes. A MASE below 1 still means "the model beats the free baseline" — the baseline is just a different one.

The business takeaway: **build the classical benchmarks first on every new series; let the data tell you which one your learned model has to beat.** A model evaluated against the wrong baseline can look impressive while adding no real value.

With the benchmarks established on both seasonal and non-seasonal series, section 8 asks: can a simple learned model — linear regression on lag features — beat those free baselines?

---

## 8. Lag Features + Linear Regression

The cheapest, most useful features for any business time series are **lags**: last month's value (`lag1`) and the value 12 months ago (`lag12`). The ACF in section 3.5 gave us the direct empirical justification — the slow decay confirmed that `lag1` carries strong momentum signal, and the peaks at lags 12 and 24 confirmed that `lag12` captures the annual seasonal cycle. A linear regression on `[lag1, lag12]` is a strong, interpretable baseline that captures both short-term momentum and annual seasonality without any deep-learning machinery. The workforce planner can read the coefficients directly: "each additional thousand employees last month contributes X thousand to next month's forecast; each additional thousand from the same month last year contributes Y."

In [ ]:
def add_lags(frame, lags=(1, 12)):
    out = frame.copy().sort_values("ds").reset_index(drop=True)
    for L in lags:
        out[f"lag{L}"] = out["y"].shift(L)
    return out

# Build lag features on the FULL series so train/val/test rows can pull lags from earlier rows
df_lag = add_lags(df).dropna()

# Re-split on the lagged frame using the same date boundaries
train_max_date = df_train["ds"].max()
val_max_date   = df_val["ds"].max()
df_lag_train = df_lag[df_lag["ds"] <= train_max_date].copy()
df_lag_val   = df_lag[(df_lag["ds"] > train_max_date) & (df_lag["ds"] <= val_max_date)].copy()
df_lag_test  = df_lag[df_lag["ds"] > val_max_date].copy()

print(f"Lagged train: n={len(df_lag_train)}  (lost {len(df_train) - len(df_lag_train)} rows to lag12)")
print(f"Lagged val  : n={len(df_lag_val)}")
print(f"Lagged test : n={len(df_lag_test)}")

X_train_lag = df_lag_train[["lag1", "lag12"]].values
y_train_lag = df_lag_train["y"].values
X_val_lag   = df_lag_val[["lag1", "lag12"]].values
y_val_lag   = df_lag_val["y"].values

lr = LinearRegression().fit(X_train_lag, y_train_lag)
y_pred_train = lr.predict(X_train_lag)
y_pred_val   = lr.predict(X_val_lag)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(df_lag_train["ds"], y_train_lag, color="#1f77b4", label="Actual", linewidth=0.8)
axes[0].plot(df_lag_train["ds"], y_pred_train, color="#2ca02c", label="Linear [lag1, lag12]", linewidth=0.8)
axes[0].set_title("Training fit")
axes[0].legend()
axes[1].plot(df_lag_val["ds"], y_val_lag, color="black", label="Actual")
axes[1].plot(df_lag_val["ds"], y_pred_val, color="#2ca02c", label="Linear [lag1, lag12]")
axes[1].set_title("Validation fit")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"\nLinear coefficients : lag1={lr.coef_[0]:.3f}, lag12={lr.coef_[1]:.3f}")
print(f"Linear intercept   : {lr.intercept_:.1f}")
print(f"Validation MAE     : {mean_absolute_error(y_val_lag, y_pred_val):.1f}")


**Reading the output:**

Two panels, one story.

**Left panel (Training fit).** The green line (model predictions) tracks the blue line (actuals) tightly across the entire training window. This is the visual confirmation that `lag1` + `lag12` carry most of the forecastable signal — the model captures both the long-run trend (via `lag1`) and the annual seasonal peaks and troughs (via `lag12`). The fit is not perfect; you can see small gaps during recession periods where the actuals drop faster than the lagged features predict. Those gaps are the structural breaks from §3.1 — the same macro shocks that appeared in the STL remainder panel.

**Right panel (Validation fit).** The green line tracks the black actuals in the validation window. This is the honest test — the model was fitted on training data only, and these validation months were never seen during fitting. If the validation fit tracks as well as the training fit, the model generalizes. If the validation fit shows wider gaps or systematic bias, the model is overfitting the training period.

**The printed coefficients** tell the workforce planner a concrete story. The `lag1` coefficient is typically near 0.8–0.9 — each thousand employees last month contributes roughly 800–900 to next month's prediction. The `lag12` coefficient is typically near 0.15–0.25 — each thousand employees from the same month last year contributes 150–250. The intercept is relatively small. These three numbers *are* the entire model, and any analyst can verify the forecast by hand: multiply last month's value by the lag1 coefficient, add last year's same-month value times the lag12 coefficient, add the intercept — done.

> **A question that often comes up here:** *"Why don't the coefficients add to 1?"* They roughly do, but not exactly — the exact sum depends on how much of next month's variance is explained by short-term momentum (`lag1`) vs. the seasonal anchor (`lag12`). When the trend is strong (as here), `lag1` dominates; in highly seasonal series with little trend, `lag12` dominates.

> **A question that often comes up at this point:** *"Shouldn't I difference the series first to make it stationary?"* That concern comes from the ARIMA tradition, where the model assumes a stationary process and differencing is the mechanism that achieves it. Here we take a different approach: a regression model with lag features in levels. The model learns the relationship between this month's value and last month's (and last year's) directly — no explicit differencing required. Both frameworks can work; the lag-feature regression is more transparent for a first encounter because the coefficients have a plain-language interpretation ("each unit of last month's employment contributes X to next month's forecast").

The single-split validation MAE gives one number, but that is one roll of the dice. Section 9 runs all five candidates — the four classical benchmarks plus the linear model — on the same walk-forward folds and applies the Student's *t* 95% CI decision rule from nb08.

---

## 9. Cross-Validated Comparison — Five Candidates, Identical Folds

The single-split fit in section 8 showed promising validation numbers, but a single split is one roll of the dice. We now compare all five forecasters — Mean, Naive, Seasonal-Naive, Linear [lag1, lag12], and Ridge [lag1, lag12] — on the **same** walk-forward folds, so every candidate faces exactly the same training and validation windows. Each fold's MAE is one observation; the mean and Student's *t* 95% CI from nb08 quantify how confident we are in the ranking. If two candidates' CIs overlap, we cannot tell them apart; if they do not overlap, we have a genuine winner.

In [ ]:
def cv_score_func(history_to_pred, df_lag_window, splits, score_fn=None):
    """Run a forecaster across walk-forward folds and return per-fold MAEs."""
    if score_fn is None:
        score_fn = mean_absolute_error
    scores = []
    for tr, va in splits.split(df_lag_window):
        train_chunk = df_lag_window.iloc[tr]
        val_chunk = df_lag_window.iloc[va]
        y_pred = history_to_pred(train_chunk, val_chunk)
        scores.append(score_fn(val_chunk["y"], y_pred))
    return np.array(scores)

def pred_naive(train, val):           return val["lag1"].values
def pred_seasonal_naive(train, val):  return val["lag12"].values
def pred_mean(train, val):            return np.full(len(val), train["y"].mean())
def pred_linear(train, val):
    m = LinearRegression().fit(train[["lag1", "lag12"]], train["y"])
    return m.predict(val[["lag1", "lag12"]])
def pred_ridge(train, val):
    m = Ridge(alpha=1.0, random_state=RANDOM_SEED).fit(train[["lag1", "lag12"]], train["y"])
    return m.predict(val[["lag1", "lag12"]])

splits = TimeSeriesSplit(n_splits=5)
candidates = {
    "Mean":             pred_mean,
    "Naive (lag1)":     pred_naive,
    "Seasonal-Naive (lag12)": pred_seasonal_naive,
    "Linear [lag1, lag12]":   pred_linear,
    "Ridge  [lag1, lag12]":   pred_ridge,
}

# --- Selection metric (MAE) with Student's t 95% CI ---
results = pd.DataFrame({name: cv_score_func(fn, df_lag_train, splits) for name, fn in candidates.items()})
t_crit = student_t.ppf(0.975, df=4)
summary = pd.DataFrame({
    "MAE_mean": results.mean(),
    "MAE_sd":   results.std(ddof=1),
    "CI_halfwidth": results.std(ddof=1) / np.sqrt(5) * t_crit,
}).sort_values("MAE_mean")
summary["CI_low"] = summary["MAE_mean"] - summary["CI_halfwidth"]
summary["CI_high"] = summary["MAE_mean"] + summary["CI_halfwidth"]
print("Selection metric (MAE) — 5-fold walk-forward CV with 95% CI:")
print(summary.round(2))

# --- Multi-metric sensitivity check (do other metrics agree on the ranking?) ---
def per_fold_all_metrics(history_to_pred, df_lag_window, splits, season=12):
    out = {"MAE": [], "RMSE": [], "MAPE": [], "MASE": []}
    for tr, va in splits.split(df_lag_window):
        train_chunk = df_lag_window.iloc[tr]
        val_chunk = df_lag_window.iloc[va]
        yt = val_chunk["y"].values
        yp = history_to_pred(train_chunk, val_chunk)
        out["MAE"].append(np.mean(np.abs(yt - yp)))
        out["RMSE"].append(np.sqrt(np.mean((yt - yp) ** 2)))
        out["MAPE"].append(np.mean(np.abs((yt - yp) / yt)) * 100.0)
        th = train_chunk["y"].values
        if len(th) > season:
            mae_naive = np.mean(np.abs(th[season:] - th[:-season]))
            out["MASE"].append(np.mean(np.abs(yt - yp)) / mae_naive)
        else:
            out["MASE"].append(np.nan)
    return {k: np.array(v) for k, v in out.items()}

all_results = {name: per_fold_all_metrics(fn, df_lag_train, splits) for name, fn in candidates.items()}
metric_means = pd.DataFrame({m: {name: r[m].mean() for name, r in all_results.items()}
                             for m in ["MAE", "RMSE", "MAPE", "MASE"]})
metric_means = metric_means.loc[summary.index]  # match selection ordering
print("\\nMulti-metric sensitivity check — per-candidate mean across all four metrics:")
print(metric_means.round(3))
print("\\nRanking under each metric (1 = best):")
print(metric_means.rank(axis=0).astype(int))

# --- Selection bar chart (MAE with 95% CI) ---
fig, ax = plt.subplots(figsize=(11, 5))
y_pos = np.arange(len(summary))
ax.barh(y_pos, summary["MAE_mean"],
        xerr=summary["CI_halfwidth"], color="#1f77b4", edgecolor="black", capsize=4)
ax.set_yticks(y_pos)
ax.set_yticklabels(summary.index)
ax.invert_yaxis()
ax.set_xlabel("MAE (5-fold walk-forward CV; bars = 95% CI)")
ax.set_title("Five-candidate forecast comparison — selection metric (MAE)")
plt.tight_layout()
plt.show()


**Reading the output:**

Three outputs, read in sequence.

**The MAE selection table** is the primary decision tool. Each row is one candidate model; the columns show the mean MAE across five walk-forward folds, the standard deviation, the CI half-width, and the lower and upper bounds of the 95% CI. The candidates are sorted by mean MAE — the top row is the current leader. Look at the CI columns: if the leader's CI does not overlap with the runner-up's CI, the leader is **genuinely better** on this metric. If the CIs overlap, you cannot distinguish them statistically — pick the simpler model.

**The multi-metric sensitivity table** shows the mean score for each candidate under all four metrics (MAE, RMSE, MAPE, MASE). The ranking table below it asks *"would I pick a different champion if I cared about a different metric?"* If all four metrics rank the same model first, the champion is **robust** — ship it with confidence. If the rankings disagree (model A wins on MAE but model B wins on RMSE), the disagreement points you to the metric whose error structure matches the business cost.

**The bar chart** makes the CI comparison visual. Horizontal bars show each candidate's mean MAE; error bars show the 95% CI. Look for gaps between bars: a clear gap with no error-bar overlap means a genuine difference; overlapping error bars mean statistically indistinguishable candidates.

Three interpretation rules borrowed from nb08:

1. **Non-overlapping CIs** between candidate A and candidate B → A is genuinely better on the selection metric.
2. **Overlapping CIs** → no statistical evidence to prefer one over the other; pick the simpler model (Occam's razor).
3. **Mean is far worse than the rest** → expected. It ignores trend and seasonality entirely; it is only here as a sanity floor.

If the linear and Ridge models have overlapping CIs, **Ridge does not earn its place** here — the regularization adds machinery without a measurable payoff. That is the right outcome for a 2-feature model; Ridge typically wins when the feature count is large and multicollinearity is a real risk.

---

## 📝 PAUSE-AND-DO Exercise 1 — Add `lag2` and `lag6` (10 minutes)

**Task:** Engineer two more lag features (`lag2`, `lag6`) and rerun the comparison. Does the four-feature linear regression beat the two-feature baseline by **non-overlapping CIs**?

**Hints:**
- Use `add_lags(df, lags=(1, 2, 6, 12))` to extend the lag list.
- Recompute `df_lag_train` from the new lagged frame (same date boundaries as before).
- Add a sixth row to the comparison table — call it `Linear [lag1, lag2, lag6, lag12]`.
- Compare its CI to the original `Linear [lag1, lag12]`. Overlap = the new features did not earn their place.

Type your code in the cell below.


> 💡 **Gemini Prompt:** *"I have a walk-forward CV comparison of five forecasters (Mean, Naive, Seasonal-Naive, Linear [lag1, lag12], Ridge [lag1, lag12]) on monthly US retail employment data, using TimeSeriesSplit(n_splits=5). The helper function add_lags(df, lags) creates lagged columns, and cv_score_func(pred_fn, df_lag_train, splits) returns per-fold MAEs. I want to add two more lag features — lag2 and lag6 — and see whether a four-feature LinearRegression beats the two-feature version. Use add_lags(df, lags=(1, 2, 6, 12)).dropna() to build the extended feature set, re-split using train_max_date, define a pred_linear_v2 function using features ['lag1', 'lag2', 'lag6', 'lag12'], run cv_score_func on the same TimeSeriesSplit folds, and build an updated MAE summary table with Student's t 95% CIs (t_crit is already defined). Print the table and a horizontal bar chart comparing all six candidates."*
>
> **After running, verify:**
> - [ ] The new model `Linear [lag1, lag2, lag6, lag12]` appears in the summary table alongside the original five candidates
> - [ ] The CI for the four-feature model overlaps (or does not overlap) with `Linear [lag1, lag12]` — note which
> - [ ] The bar chart shows error bars for all six candidates
> - [ ] No test-set data was used anywhere

In [ ]:
# YOUR SOLUTION CODE HERE

# Hints:
# df_lag_v2 = add_lags(df, lags=(1, 2, 6, 12)).dropna()
# df_lag_v2_train = df_lag_v2[df_lag_v2["ds"] <= train_max_date]
# def pred_linear_v2(train, val):
#     features = ["lag1", "lag2", "lag6", "lag12"]
#     m = LinearRegression().fit(train[features], train["y"])
#     return m.predict(val[features])
# results["Linear [lag1, lag2, lag6, lag12]"] = cv_score_func(pred_linear_v2, df_lag_v2_train, splits)
# Build the new summary, compare CIs.


## 10. Opening the Locked Test Window — One-Shot Evaluation

We now do the time-series analog of nb14’s “test-set opening ceremony.” The ritual is the same: pick the champion, refit on all of train + val, predict the locked window once, and read the verdict — **INSIDE / ABOVE / BELOW** the CV 95% CI. What is new here is the **prediction interval**: we estimate the residual sigma from walk-forward folds (not from the final training fit) and wrap a 95% Gaussian band around each point forecast. The workforce planner gets not just “forecast = X” but “forecast = X ± Y with Z% empirical coverage” — a deliverable the legislature can read.

In [ ]:
# Step 1: estimate residual sigma from walk-forward training fits.
# Each fold's residuals come from a fit that did NOT see the validation rows.
fold_residuals = []
for tr, va in splits.split(df_lag_train):
    train_chunk = df_lag_train.iloc[tr]
    val_chunk   = df_lag_train.iloc[va]
    m_fold = LinearRegression().fit(train_chunk[["lag1", "lag12"]], train_chunk["y"])
    pred_fold = m_fold.predict(val_chunk[["lag1", "lag12"]])
    fold_residuals.extend(val_chunk["y"].values - pred_fold)
fold_residuals = np.array(fold_residuals)
sigma_residual = fold_residuals.std(ddof=1)
print(f"Walk-forward residual sigma: {sigma_residual:.2f} (units: thousands of employees)")

# Step 2: refit champion on train + val (lag features only — no test-set leak)
df_lag_trainval = pd.concat([df_lag_train, df_lag_val])
champion = LinearRegression().fit(
    df_lag_trainval[["lag1", "lag12"]], df_lag_trainval["y"]
)

# Step 3: point forecast on the locked test window
y_test_pred = champion.predict(df_lag_test[["lag1", "lag12"]])
test_mae = mean_absolute_error(df_lag_test["y"], y_test_pred)

# Step 4: 95% prediction interval (Gaussian assumption on residuals)
z_95 = 1.96
y_test_lower = y_test_pred - z_95 * sigma_residual
y_test_upper = y_test_pred + z_95 * sigma_residual

# Step 5: empirical coverage of the 95% PI on the locked test window
inside = ((df_lag_test["y"].values >= y_test_lower) &
          (df_lag_test["y"].values <= y_test_upper)).mean()

# Pull the champion's CV CI for the verdict
champ_row = summary.loc["Linear [lag1, lag12]"]
cv_low, cv_high = champ_row["CI_low"], champ_row["CI_high"]
verdict = ("INSIDE the CV 95% CI" if cv_low <= test_mae <= cv_high
           else "ABOVE the CV 95% CI (overfitting?)" if test_mae > cv_high
           else "BELOW the CV 95% CI (lucky test window?)")

print(f"\\nChampion: Linear [lag1, lag12]")
print(f"CV MAE 95% CI : [{cv_low:.2f}, {cv_high:.2f}]")
print(f"Test MAE      : {test_mae:.2f}  ->  {verdict}")
print(f"Empirical 95% PI coverage on test: {inside*100:.1f}%  (nominal: 95.0%)")

# Plot: train + val + test_actual + champion_forecast + shaded 95% PI band
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(df_lag_train["ds"], df_lag_train["y"], color="#1f77b4", label="Train", linewidth=0.8)
ax.plot(df_lag_val["ds"],   df_lag_val["y"],   color="#ff7f0e", label="Val", linewidth=0.8)
ax.plot(df_lag_test["ds"],  df_lag_test["y"],  color="#d62728", label="Test (actual)", linewidth=1.5)
ax.plot(df_lag_test["ds"],  y_test_pred,       color="#2ca02c", linestyle="--",
        label="Champion forecast", linewidth=1.5)
ax.fill_between(df_lag_test["ds"], y_test_lower, y_test_upper,
                color="#2ca02c", alpha=0.20, label="95% prediction interval")
ax.set_title("Locked Test Window — Forecast + 95% Prediction Interval")
ax.legend()
plt.tight_layout()
plt.show()


**Reading the output:**

Three outputs, one verdict.

**The printed summary** shows the champion name, the CV MAE 95% CI (from §9), the test MAE, and the verdict. The verdict follows nb14's protocol:

- **INSIDE the CV 95% CI** means the walk-forward CV estimate generalized to the truly unseen test window. The CV-based selection was honest — the workforce planner can quote the test MAE with confidence.
- **ABOVE the CV 95% CI** signals that the champion performed worse on the test window than the CV predicted — possible overfitting to the training history, or a structural break in the test period (a recession the model could not foresee).
- **BELOW the CV 95% CI** means the champion performed better than expected — the test window was easier than the average CV fold. Unusual but not alarming.

**The empirical coverage** is the new diagnostic. We constructed the 95% PI under a **Gaussian assumption** on the walk-forward residuals: $\hat{y}_t \pm 1.96 \cdot \sigma_{\text{residual}}$. Then we asked *"what fraction of locked-test actuals actually fell inside that interval?"*. Three reading rules:

- **Coverage near 95%** (say, 90–98%): the Gaussian assumption holds, the interval is honest, and the workforce planner can quote it to the legislature.
- **Coverage well below 95%** (say, 70–85%): the model is **overconfident** — the residuals have heavier tails than Gaussian, or there are structural breaks the lag-feature model cannot capture (recessions, policy shocks). The interval needs to be widened before reporting.
- **Coverage near 100%**: the interval is **too wide** — typically because the residual sigma was inflated by a few outlier folds. Tighter intervals would still cover the right amount and be more useful to a decision-maker.

**The plot** shows the full series in three colors (blue train, orange val, red test actuals) plus the green dashed champion forecast and a shaded green band for the 95% prediction interval. Look for whether the red test actuals stay inside the green band — that is the coverage check visualized. Any point where the red line escapes the band is a moment the model's uncertainty estimate was too narrow.

For the workforce planner, the deliverable is no longer just *"forecast = X"* but *"forecast = X, 95% interval [X\u2212Y, X+Y], and the model has been cross-validated to deliver that coverage on held-out data."* That is the line that goes on the M4 poster.

> **A question that often comes up here:** *"Why use the residual sigma from CV folds instead of from the final training fit?"* Because the final training residuals are in-sample — the model fitted those rows. Walk-forward residuals are out-of-sample; they reflect the noise the model will encounter on truly future data. Using in-sample residuals would systematically underestimate sigma and produce overconfident intervals. Same principle as nb08's CV CIs.

---

## 📝 PAUSE-AND-DO Exercise 2 — Add Ridge Tuning (10 minutes)

**Task:** Ridge with `alpha=1.0` may be over- or under-regularized. Sweep `alpha ∈ [0.01, 0.1, 1, 10, 100]`, run walk-forward CV at each alpha, and pick the alpha that minimizes mean MAE. Compare its CI to the unregularized linear baseline.

**Hints:**
- Loop over alphas; each iteration runs `cv_score_func(...)` with a new `pred_ridge_alpha`.
- Build a small `pd.DataFrame` of `alpha` vs. `MAE_mean` and `CI_halfwidth`.
- The plot to make: alpha on a log x-axis, MAE on the y-axis, with error bars for the CI.

Type your code in the cell below.

**Bonus — interpret the prediction-interval coverage:** look at the empirical coverage printed in section 10 (`X.X%`). Is it close to the nominal 95%? If not, write one sentence explaining what that says about the residual distribution (heavy tails? structural break? Gaussian approximation failing?).


> 💡 **Gemini Prompt:** *"I have a walk-forward CV setup using TimeSeriesSplit(n_splits=5) on monthly US retail employment data with lag1 and lag12 features in df_lag_train. The helper cv_score_func(pred_fn, df_lag_train, splits) returns per-fold MAEs, and t_crit is already defined. Sweep Ridge alpha over [0.01, 0.1, 1, 10, 100] — at each alpha, define a prediction function that fits Ridge(alpha=alpha, random_state=RANDOM_SEED) on ['lag1', 'lag12'], run cv_score_func, and collect the mean MAE and CI half-width. Build a DataFrame of alpha vs MAE_mean and CI_halfwidth. Print the table, plot alpha on a log x-axis vs MAE with error bars using ax.errorbar, and compare the best Ridge CI to the unregularized LinearRegression CI from the summary table to determine whether Ridge earns its place."*
>
> **After running, verify:**
> - [ ] The table shows five rows (one per alpha) with MAE_mean and CI half-width columns
> - [ ] The plot has alpha on a log-scaled x-axis with error bars at each point
> - [ ] A printed comparison states whether the best Ridge CI overlaps with the Linear CI
> - [ ] All evaluation uses walk-forward CV on training data only — no test-set leak

In [ ]:
# YOUR SOLUTION CODE HERE

# Hints:
# alphas = [0.01, 0.1, 1, 10, 100]
# rows = []
# for a in alphas:
#     def pred_ridge_a(train, val, alpha=a):
#         m = Ridge(alpha=alpha, random_state=RANDOM_SEED).fit(train[["lag1","lag12"]], train["y"])
#         return m.predict(val[["lag1","lag12"]])
#     fold_maes = cv_score_func(pred_ridge_a, df_lag_train, splits)
#     rows.append({"alpha": a, "MAE_mean": fold_maes.mean(), "CI_hw": fold_maes.std(ddof=1)/np.sqrt(5)*t_crit})
# Then plot with errorbar() on a log-x axis.


## 11. Forecast Accuracy Diagnostics — Residuals and Horizon

Two questions are worth answering before we wrap up. First: **why did we estimate the prediction-interval sigma from walk-forward residuals instead of in-sample training residuals?** A side-by-side comparison answers it visually. Second: **does forecast error grow as we predict further ahead?** A rolling-forecast-origin sweep across horizons answers that one directly.

These two diagnostics complete the toolkit a workforce planner needs to defend a forecast: the **point forecast** (§7-9), the **prediction interval** (§10), the **residual diagnostic** (§11.1), and the **horizon curve** (§11.2).


### 11.1 In-sample residuals vs walk-forward residuals

If the champion is fit on training data and we measure its residuals on those *same* rows, we get **in-sample** residuals — the errors from data the model has already seen and optimized against. Those residuals are systematically smaller than residuals on truly unseen rows, because the model has "memorized" some of the training noise. The walk-forward residuals, by contrast, come from validation windows the model never saw during fitting — they reflect what the model will actually face on future data. The gap between the two is exactly the gap a prediction interval has to honor. If the workforce planner uses in-sample sigma to build a 95% PI, the interval will be too narrow and more than 5% of future observations will fall outside. That is why §10's PI used walk-forward residual sigma.

In [ ]:
# Champion fit on the full lagged training data (the "in-sample" world)
champ_in_sample = LinearRegression().fit(
    df_lag_train[["lag1", "lag12"]], df_lag_train["y"]
)
in_sample_pred = champ_in_sample.predict(df_lag_train[["lag1", "lag12"]])
in_sample_residuals = df_lag_train["y"].values - in_sample_pred
in_sample_mae = float(np.mean(np.abs(in_sample_residuals)))
in_sample_sigma = float(in_sample_residuals.std(ddof=1))

# Walk-forward residuals already computed in §10 as `fold_residuals`,
# `sigma_residual` — reuse them.
cv_mae = float(np.mean(np.abs(fold_residuals)))
cv_sigma = float(sigma_residual)

cmp_table = pd.DataFrame({
    "MAE":             [in_sample_mae, cv_mae],
    "Residual sigma":  [in_sample_sigma, cv_sigma],
}, index=["In-sample (training fit)", "Walk-forward CV (out-of-sample)"])
print("Residual diagnostics — same champion, two ways of measuring its noise:")
print(cmp_table.round(2))
print(f"\nRatio (CV / in-sample) MAE   : {cv_mae / in_sample_mae:.2f}x")
print(f"Ratio (CV / in-sample) sigma : {cv_sigma / in_sample_sigma:.2f}x")

# Two-panel plot: residual histograms overlaid + MAE bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(in_sample_residuals, bins=40, color="#1f77b4", alpha=0.6, label="In-sample")
axes[0].hist(fold_residuals,      bins=40, color="#d62728", alpha=0.6, label="Walk-forward CV")
axes[0].axvline(0, color="black", linewidth=0.5)
axes[0].set_xlabel("Residual (units: thousands of employees)")
axes[0].set_ylabel("Count")
axes[0].set_title("Residual distributions")
axes[0].legend()

axes[1].bar(["In-sample\\n(training fit)", "Walk-forward CV\\n(out-of-sample)"],
            [in_sample_mae, cv_mae],
            color=["#1f77b4", "#d62728"], edgecolor="black")
axes[1].set_ylabel("Mean absolute residual")
axes[1].set_title("Residual MAE — why the PI used CV, not in-sample")
plt.tight_layout()
plt.show()


**Reading the output:**

Two panels side by side make the case visually.

**Left panel (Residual histograms).** The blue histogram shows in-sample residuals — the errors from the champion fitted on training data and evaluated on those same training rows. The red histogram shows walk-forward CV residuals — the errors from validation windows the model never saw during fitting. The red histogram is **wider and more dispersed** than the blue one. The in-sample residuals cluster tightly around zero because the model optimized against those exact rows; the walk-forward residuals spread further because the model faces unseen dynamics — seasonal shifts it did not train on, trend changes it could not anticipate.

**Right panel (MAE bar chart).** Two bars compare the mean absolute residual from each source. The walk-forward CV MAE is typically **1.5\u00d7 to 3\u00d7 larger** than the in-sample MAE on a series with strong autocorrelation like this one. That multiplier is the honesty gap — the difference between how good the model *looks* on data it has seen and how good it *actually is* on data it has not.

**Why this matters for the workforce planner:** §10's prediction interval used the walk-forward sigma (the red histogram's spread), not the in-sample sigma (the blue histogram's spread). If we had used the in-sample sigma, the 95% prediction interval would have been **systematically too narrow** — claiming 95% coverage on paper while letting more than 5% of true future values fall outside the band. The workforce planner would have quoted a tight interval to the legislature, and then been embarrassed when actual employment repeatedly landed outside it.

This is the time-series version of the lesson nb08 taught for k-fold: **in-sample evaluation is overconfident; out-of-sample evaluation is honest.** The same principle applies to point estimates (MAE) and to uncertainty estimates (sigma).

### 11.2 Forecast horizon and accuracy

A workforce planner who asks *"what will retail employment be next month?"* gets a tight answer. A planner who asks *"what about next year?"* gets a much wider band — forecast error compounds with distance. To quantify this, we sweep horizon $h \in \{1, 2, \dots, 12\}$ using a **rolling forecast origin** (each `TimeSeriesSplit` fold supplies one cutoff) combined with **recursive multi-step forecasting**: the model predicts month 1, feeds that prediction back as `lag1` to predict month 2, feeds month 2 back to predict month 3, and so on. Each step uses the model's own imperfect output rather than actual data, so errors accumulate.

In [ ]:
def recursive_forecast(model, history_y, h, season=12):
    """Forecast h steps ahead recursively. `history_y` must contain at least
    the last `season` observed values; predictions feed back as lag1 inputs."""
    history = list(history_y)
    preds = []
    for _ in range(h):
        lag1  = history[-1]
        lag12 = history[-season]
        x = np.array([[lag1, lag12]])
        yhat = float(model.predict(x)[0])
        preds.append(yhat)
        history.append(yhat)
    return np.array(preds)

# For each TS-CV fold, fit on training, recursive-forecast h=1..12,
# collect squared errors per horizon.
HORIZONS = list(range(1, 13))
errors_by_h = {h: [] for h in HORIZONS}

for tr, va in TimeSeriesSplit(n_splits=5).split(df_lag_train):
    train_chunk = df_lag_train.iloc[tr]
    val_chunk   = df_lag_train.iloc[va]
    m = LinearRegression().fit(train_chunk[["lag1", "lag12"]], train_chunk["y"])
    history = train_chunk["y"].values  # actual history up to the cutoff
    h_max = min(len(HORIZONS), len(val_chunk))
    preds = recursive_forecast(m, history, h_max)
    actuals = val_chunk["y"].values[:h_max]
    for i, (a, p) in enumerate(zip(actuals, preds), start=1):
        if i in errors_by_h:
            errors_by_h[i].append((a - p) ** 2)

rmse_by_h = {h: float(np.sqrt(np.mean(es))) for h, es in errors_by_h.items() if es}

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(rmse_by_h.keys()), list(rmse_by_h.values()),
        "o-", color="#9467bd", linewidth=2, markersize=8)
ax.set_xlabel("Forecast horizon h (months ahead)")
ax.set_ylabel("RMSE (recursive forecast)")
ax.set_title("Forecast accuracy degrades with horizon — Linear [lag1, lag12]")
ax.grid(alpha=0.3)
ax.set_xticks(list(rmse_by_h.keys()))
plt.tight_layout()
plt.show()

rmse_table = pd.Series(rmse_by_h, name="RMSE").round(2).to_frame()
rmse_table.index.name = "h (months ahead)"
print(rmse_table)


**Reading the output:**

The plot shows a single line: RMSE on the y-axis, forecast horizon (months ahead) on the x-axis, from 1 to 12.

**The line rises monotonically.** The 1-month-ahead RMSE is the smallest — the model's prediction for next month is its most accurate. By the time you reach 12 months ahead, the RMSE is typically **3\u00d7 to 5\u00d7 larger** than the 1-month value. The table below the plot prints the exact RMSE at each horizon so the workforce planner can quote specific numbers: "our 1-month forecast is accurate to within \u00b1X thousand employees; our 12-month forecast is accurate to within \u00b1Y thousand."

**Two effects compound to produce this degradation.** **Recursive feedback** is the first: for horizons beyond one month, the model uses its own imperfect predictions as `lag1` inputs. The month-1 prediction feeds into the month-2 prediction, the month-2 prediction feeds into month-3, and so on. Each step's error becomes part of the next step's input, so small mistakes at early horizons accumulate into larger mistakes at later horizons. **Information staleness** is the second: the model has access to actual lag values up to the forecast origin, but no information about events after that point. The further out it forecasts, the more events it cannot know about — recessions, policy shocks, structural changes in the retail labor market.

**Practical implication for the workforce planner:** the forecast quoted **one month out** can carry a tight prediction interval; the forecast quoted **one year out** cannot. When the legislature asks for a 12-month forecast, the planner should present it alongside this horizon-vs-RMSE curve and explicitly note the widening uncertainty. This curve is a strong candidate figure for the M4 poster's "Limitations" section — it demonstrates intellectual honesty about what the model can and cannot do.

> **A question that often comes up here:** *"Why does this differ from the \u00a79 CV table?"* The \u00a79 table averages across all rows in each validation fold, so it implicitly averages over many horizons mixed together. \u00a711.2 separates them — one RMSE per horizon — which is what you actually need when the business question is *"how far ahead can we trust this forecast?"*

With the point forecast, the prediction interval, the residual diagnostic, and the horizon curve all in hand, you have the full toolkit the workforce planner needs to defend a forecast. Section 12 pulls it all together.

---

## 12. Wrap-Up — Key Takeaways

1. **Forecasting is supervised learning with one structural rule: never let the future leak into the past.** That single rule changes the train/test split (recent slice held out), the cross-validation strategy (`TimeSeriesSplit`), and what counts as a feature (lags, not random shuffling).
2. **The Week-1 analytics workflow ports cleanly to time series.** EDA → split → baselines → linear features → regularization is the same recipe; only the partition strategy and feature engineering change.
3. **Naive baselines are surprisingly hard to beat.** If your fancy model does not beat seasonal-naive on identical CV folds with non-overlapping CIs, you do not have a champion — you have noise.
4. **The cost of lag features is the loss of the earliest rows.** A 12-month seasonal lag costs you the first year of history. Plan for it.
5. **Walk-forward CV is the time-series spine of CV-first evaluation,** exactly like `StratifiedKFold` was the classification spine in nb08–nb14.

### Beyond This Introduction

This notebook is an introduction — enough to build, evaluate, and defend a lag-feature linear forecast on a real business series. A dedicated time-series forecasting course covers substantially more ground:

| Topic | What it adds |
|---|---|
| **ARIMA / SARIMA** | The classical Box-Jenkins approach: differencing for stationarity, ACF/PACF-based order selection, seasonal terms. Still the benchmark in many industries. |
| **Exponential Smoothing (ETS)** | Holt-Winters and state-space models that weight recent observations more heavily than distant ones. The go-to for short-term inventory and demand planning. |
| **Prophet** | Meta's decomposable model with built-in holiday effects and automatic changepoint detection. Popular in e-commerce and retail forecasting. |
| **Multiple Seasonalities** | Series with daily, weekly, *and* annual cycles simultaneously — hourly electricity demand, web traffic, call-center staffing. |
| **Multivariate Forecasting (VAR)** | Using multiple related series (employment, GDP, consumer confidence) to forecast each other. Includes Granger causality — testing whether one series actually *predicts* another. |
| **Deep Learning (RNN / LSTM / Transformer)** | Sequence models that learn non-linear temporal dependencies from very long histories. Practical when you have thousands of series and large compute budgets. |
| **Hierarchical Reconciliation** | Forecasting at store, region, and national level simultaneously and reconciling the numbers to be consistent. Essential for retail chains and government agencies. |
| **Conformal Prediction Intervals** | Distribution-free uncertainty bands that guarantee coverage *without* the Gaussian assumption we used in §10. The fix when your residuals have heavy tails. |

The lag-feature regression you built today is not a toy — it is genuinely competitive on monthly business series with moderate trend and seasonality. The tools above extend the toolkit when the series is longer, more complex, or demands richer uncertainty quantification.

> **A question that often comes up here:** *"Where do RNNs and transformers fit?"* They are alternatives to lag-feature linear models when (a) the series is long enough (thousands of points, not 960), (b) the dependence is highly non-linear, and (c) you can spare an order of magnitude more compute. For business problems with a few decades of monthly history, a well-engineered lag-feature linear regression is almost always the right starting point — and often the right ending point. Deep learning gets the awareness module it deserves in **nb19**.

**Next stop — nb17: Data Communication and Poster Design.** Now that you have a forecast, a defensible CV-based comparison, and a clean test-set ceremony verdict, the question becomes how to **communicate** them: the six principles of data communication, the eleven-section poster architecture for the M4 deliverable, and the data-ink-ratio cleanup that turns a notebook plot into a poster figure.

---

## Participation Assignment Submission Instructions

1. **Complete both PAUSE-AND-DO exercises** (sections after 9 and 10).
2. **Run all cells** (`Runtime → Run all`).
3. **Save with output** (`File → Download → Download .ipynb`).
4. **Submit to Brightspace** as `nb16_time_series_forecasting_<your_lastname>.ipynb`.

**Bibliography**
- Hyndman & Athanasopoulos: *Forecasting: Principles and Practice* (FPP3) — the [free online textbook](https://otexts.com/fpp3/) is the deep dive on every concept above.
- scikit-learn User Guide: `TimeSeriesSplit` and time-series cross-validation.
- statsmodels: `STL` decomposition and the autocorrelation function.

<center>

# Thank you!

</center>
